# Sampling Strategies for Language Model Generation

A language model produces a probability distribution over the entire vocabulary at each generation step.
How we **sample** from that distribution determines the quality, diversity, and creativity of the output.

Greedy decoding (always picking the most probable token) produces repetitive, dull text.
Sampling introduces randomness, but uncontrolled randomness produces incoherent text.
The three core strategies -- **temperature scaling**, **top-k**, and **top-p (nucleus)** -- give us
fine-grained control over the diversity/quality tradeoff.

In this notebook we will:
1. Visualize how each strategy reshapes the probability distribution
2. Measure entropy and sparsity to quantify the effect
3. Run ablation studies over hyperparameter ranges
4. Compare empirical sample distributions across methods

## 1. Imports and Setup

In [ ]:
%matplotlib inline

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from sampler import TokenSampler, SamplingAnalyzer
from sampling import sample

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

torch.manual_seed(42)
np.random.seed(42)

## 2. Generate Synthetic Logits

We use `SamplingAnalyzer` to create a realistic logit vector over a 1000-token vocabulary.
In practice these logits come from the final linear layer of a transformer, but synthetic logits
let us study sampling in isolation.

In [ ]:
analyzer = SamplingAnalyzer(vocab_size=1000, seed=42)
logits = analyzer.generate_logits(distribution='normal')

print(f"Logit shape: {logits.shape}")
print(f"Logit range: [{logits.min().item():.3f}, {logits.max().item():.3f}]")
print(f"Logit mean:  {logits.mean().item():.3f}")
print(f"Logit std:   {logits.std().item():.3f}")

### Raw Probability Distribution

Before any sampling strategy is applied, we convert logits to probabilities via softmax.
Most tokens have negligible probability; a small handful dominate.

In [ ]:
probs = F.softmax(logits, dim=-1)
sorted_probs, sorted_indices = torch.sort(probs, descending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Full distribution
axes[0].bar(range(len(probs)), probs.numpy(), width=1.0, alpha=0.7)
axes[0].set_xlabel('Token ID')
axes[0].set_ylabel('Probability')
axes[0].set_title('Raw Probability Distribution (all 1000 tokens)')

# Sorted top-50
axes[1].bar(range(50), sorted_probs[:50].numpy(), alpha=0.7, color='coral')
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Probability')
axes[1].set_title('Top-50 Tokens by Probability (sorted)')

plt.tight_layout()
plt.show()

print(f"Top-1 probability:  {sorted_probs[0].item():.4f}")
print(f"Top-10 cumulative:  {sorted_probs[:10].sum().item():.4f}")
print(f"Top-50 cumulative:  {sorted_probs[:50].sum().item():.4f}")

The long tail is characteristic of natural language distributions. Most of the probability mass
is concentrated in a small number of tokens, but the tail is very long.

---
## 3. Temperature Sampling

Temperature divides the logits before softmax: `softmax(logits / T)`.

| Temperature | Effect |
|---|---|
| T -> 0 | Approaches greedy (argmax) -- very peaked |
| T = 1.0 | Original distribution unchanged |
| T > 1.0 | Flattens the distribution -- more random |

In [ ]:
temperatures = [0.1, 0.5, 1.0, 2.0]

print("Temperature sampling demo (each call samples one token):")
print("-" * 55)
for temp in temperatures:
    token_id = TokenSampler.temperature_sampling(logits, temperature=temp)
    token_prob = F.softmax(logits / temp, dim=-1)[token_id].item()
    entropy = TokenSampler.compute_entropy(logits / temp)
    print(f"  T={temp:<4} -> token={token_id.item():>4}, "
          f"prob={token_prob:.4f}, entropy={entropy:.3f}")

At low temperature the sampler almost always picks the highest-probability token.
At high temperature the distribution flattens out, and less-likely tokens have a real chance of being selected.

### Visualize Probability Distributions at Different Temperatures

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for ax, temp, color in zip(axes, temperatures, colors):
    scaled_probs = F.softmax(logits / temp, dim=-1)
    sorted_p, _ = torch.sort(scaled_probs, descending=True)
    ax.bar(range(50), sorted_p[:50].numpy(), alpha=0.8, color=color)
    ax.set_title(f'T = {temp}')
    ax.set_xlabel('Rank')
    ax.set_ylabel('Probability')

plt.suptitle('Sorted Top-50 Probabilities at Different Temperatures', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay comparison on a single plot (top-20 for clarity)
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(20)
width = 0.2

for i, (temp, color) in enumerate(zip(temperatures, colors)):
    scaled_probs = F.softmax(logits / temp, dim=-1)
    sorted_p, _ = torch.sort(scaled_probs, descending=True)
    ax.bar(x + i * width, sorted_p[:20].numpy(), width=width,
           label=f'T={temp}', color=color, alpha=0.85)

ax.set_xlabel('Rank')
ax.set_ylabel('Probability')
ax.set_title('Temperature Effect on Top-20 Token Probabilities')
ax.legend()
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(x)
plt.tight_layout()
plt.show()

The overlay makes the tradeoff very clear: low temperature concentrates nearly all mass on the top token,
while high temperature spreads it across many candidates.

---
## 4. Top-K Sampling

Top-k sampling keeps only the **k most probable tokens** and redistributes probability among them.
Everything outside the top-k is set to zero probability.

- k=1 is equivalent to greedy decoding
- Large k approaches unfiltered sampling

In [ ]:
k_values = [1, 5, 10, 50]

print("Top-K sampling demo:")
print("-" * 55)
for k in k_values:
    token_id = TokenSampler.top_k_sampling(logits, k=k, temperature=1.0)
    print(f"  k={k:<4} -> sampled token={token_id.item():>4}")

### Visualize Sorted Probabilities with Top-K Cutoff Lines

In [ ]:
sorted_probs, _ = torch.sort(F.softmax(logits, dim=-1), descending=True)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(100), sorted_probs[:100].numpy(), alpha=0.6, color='steelblue',
       label='Token probability')

cutoff_colors = ['red', 'orange', 'green', 'purple']
for k, color in zip(k_values, cutoff_colors):
    ax.axvline(x=k - 0.5, color=color, linestyle='--', linewidth=2,
               label=f'k={k} cutoff')

ax.set_xlabel('Rank (sorted by probability)')
ax.set_ylabel('Probability')
ax.set_title('Top-K Truncation: Only tokens left of the cutoff survive')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Show how much probability mass each k captures
k_range = [1, 2, 5, 10, 20, 50, 100, 200, 500]
cumulative_mass = [sorted_probs[:k].sum().item() for k in k_range]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_range, cumulative_mass, 'o-', color='steelblue', linewidth=2)
ax.axhline(y=0.9, color='red', linestyle=':', alpha=0.7, label='90% mass')
ax.set_xlabel('K')
ax.set_ylabel('Cumulative Probability Mass')
ax.set_title('Probability Mass Captured by Top-K')
ax.set_xscale('log')
ax.legend()
plt.tight_layout()
plt.show()

A fixed k is a blunt instrument: for peaked distributions k=10 may include irrelevant tokens,
while for flat distributions it may exclude too many plausible ones.
This motivates top-p sampling, which adapts the cutoff to the shape of the distribution.

---
## 5. Top-P (Nucleus) Sampling

Top-p sampling keeps the **smallest set of tokens whose cumulative probability >= p**.
The number of active tokens adapts to the distribution:
- Peaked distribution -> few tokens
- Flat distribution -> many tokens

This is often preferred in practice because it naturally adapts to context.

In [ ]:
p_values = [0.5, 0.9, 0.95]

print("Top-P (nucleus) sampling demo:")
print("-" * 60)
for p in p_values:
    token_id = TokenSampler.top_p_sampling(logits, p=p, temperature=1.0)
    # Count how many tokens are in the nucleus
    sorted_p, _ = torch.sort(F.softmax(logits, dim=-1), descending=True)
    cumsum = torch.cumsum(sorted_p, dim=-1)
    n_tokens = (cumsum <= p).sum().item() + 1  # +1 for the token that crosses p
    print(f"  p={p:<5} -> sampled token={token_id.item():>4}, "
          f"nucleus size={n_tokens} tokens")

### Visualize Cumulative Probability with Cutoffs

In [ ]:
sorted_probs, _ = torch.sort(F.softmax(logits, dim=-1), descending=True)
cumsum_probs = torch.cumsum(sorted_probs, dim=-1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: cumulative probability curve with cutoffs
ax = axes[0]
ax.plot(range(200), cumsum_probs[:200].numpy(), linewidth=2, color='steelblue')
p_colors = ['red', 'orange', 'green']
for p, color in zip(p_values, p_colors):
    n_tokens = (cumsum_probs <= p).sum().item() + 1
    ax.axhline(y=p, color=color, linestyle='--', alpha=0.7, label=f'p={p} ({n_tokens} tokens)')
    ax.axvline(x=n_tokens, color=color, linestyle=':', alpha=0.5)
ax.set_xlabel('Number of Tokens (sorted by probability)')
ax.set_ylabel('Cumulative Probability')
ax.set_title('Cumulative Distribution with Top-P Cutoffs')
ax.legend()

# Right: nucleus sizes for different p values
ax = axes[1]
p_sweep = np.arange(0.1, 1.0, 0.05)
nucleus_sizes = []
for p in p_sweep:
    n = (cumsum_probs <= p).sum().item() + 1
    nucleus_sizes.append(n)
ax.plot(p_sweep, nucleus_sizes, 'o-', color='coral', linewidth=2)
ax.set_xlabel('P threshold')
ax.set_ylabel('Nucleus size (# tokens)')
ax.set_title('Nucleus Size vs P Threshold')

plt.tight_layout()
plt.show()

Notice the non-linear relationship: small increases in p near 0.9 can dramatically increase the nucleus size,
because the tail tokens each contribute very little probability.

---
## 6. Entropy and Sparsity Analysis

**Entropy** measures uncertainty: higher entropy means more uniform (more random) sampling.
**Sparsity** measures what fraction of tokens have negligible probability (below 1%).

These metrics let us compare strategies quantitatively.

In [ ]:
print("Entropy and Sparsity for raw logits at different temperatures:")
print("=" * 60)
for temp in [0.1, 0.5, 1.0, 2.0, 5.0]:
    scaled = logits / temp
    entropy = TokenSampler.compute_entropy(scaled)
    sparsity = TokenSampler.compute_sparsity(scaled)
    print(f"  T={temp:<4}  entropy={entropy:>6.3f}  sparsity={sparsity:.3f}")

In [ ]:
# Compare entropy across all three methods
print("\nMethod comparison at default settings:")
print("=" * 60)

# Baseline
base_entropy = TokenSampler.compute_entropy(logits)
base_sparsity = TokenSampler.compute_sparsity(logits)
print(f"  Raw (T=1.0):   entropy={base_entropy:.3f}, sparsity={base_sparsity:.3f}")

# Temperature = 0.5
entropy_t05 = TokenSampler.compute_entropy(logits / 0.5)
sparsity_t05 = TokenSampler.compute_sparsity(logits / 0.5)
print(f"  Temp=0.5:      entropy={entropy_t05:.3f}, sparsity={sparsity_t05:.3f}")

# Top-K = 10 (approximate entropy by masking)
top_k_logits, top_k_idx = torch.topk(logits, 10)
masked_logits = torch.full_like(logits, float('-inf'))
masked_logits.scatter_(0, top_k_idx, top_k_logits)
entropy_k10 = TokenSampler.compute_entropy(masked_logits)
print(f"  Top-K=10:      entropy={entropy_k10:.3f}")

# Top-P = 0.9 (approximate)
probs = F.softmax(logits, dim=-1)
sorted_p, sorted_idx = torch.sort(probs, descending=True)
cumsum = torch.cumsum(sorted_p, dim=-1)
mask = cumsum > 0.9
mask[0] = False
p_logits = logits.clone()
p_logits[sorted_idx[mask]] = float('-inf')
entropy_p09 = TokenSampler.compute_entropy(p_logits)
print(f"  Top-P=0.9:     entropy={entropy_p09:.3f}")

Lower entropy means more deterministic sampling. Temperature and top-k/top-p
are complementary tools: temperature reshapes the distribution globally, while
top-k and top-p truncate the tail.

---
## 7. Ablation Studies (Comprehensive)

The `SamplingAnalyzer.plot_ablations()` method sweeps across hyperparameter ranges
and plots six panels:
- Temperature vs entropy, max probability, sparsity
- Top-K vs entropy
- Top-P vs entropy and active token count

In [ ]:
analyzer.plot_ablations()

### Ablation takeaways

- **Temperature vs Entropy**: Nearly linear -- doubling T roughly doubles entropy.
- **Temperature vs Max Probability**: Sharp drop-off. Even T=0.5 drastically reduces the top token's dominance.
- **Temperature vs Sparsity**: At low T, almost all tokens fall below the 1% threshold. At high T, sparsity drops as probability mass spreads out.
- **Top-K vs Entropy**: Logarithmic growth in entropy as k increases.
- **Top-P vs Entropy/Tokens**: Entropy and active tokens both rise sharply as p approaches 1.0.

---
## 8. Sample Distribution Comparison

Drawing 5000 samples from each strategy lets us see the empirical distribution.
This reveals the practical effect of each method on generation diversity.

In [ ]:
analyzer.compare_sampling_distributions()

- **T=0.1**: Samples collapse to a handful of token IDs (nearly deterministic)
- **T=1.0**: Broader spread, but still concentrated around the high-probability region
- **Top-K=10**: Exactly 10 distinct token IDs appear, enforcing strict truncation
- **Top-P=0.9**: Adaptive set -- more tokens than top-k=10, but still avoids the deep tail

---
## 9. Compact Sampler: `sampling.sample()`

The `sample()` function from `sampling.py` provides a minimal, production-style implementation
combining temperature and optional top-k in a single call.

In [ ]:
print("Compact sampler demos:")
print("=" * 50)

# Default: T=1.0, no top-k
token = sample(logits, temp=1.0, top_k=None)
print(f"  Default (T=1.0, no top-k): token={token.item()}")

# Low temperature
token = sample(logits, temp=0.3, top_k=None)
print(f"  Low temp (T=0.3):          token={token.item()}")

# Top-K with temperature
token = sample(logits, temp=0.8, top_k=10)
print(f"  T=0.8, top-k=10:           token={token.item()}")

# Very restricted
token = sample(logits, temp=0.5, top_k=3)
print(f"  T=0.5, top-k=3:            token={token.item()}")

In [ ]:
# Compare compact sampler distribution to TokenSampler
n_samples = 2000

compact_samples = [sample(logits, temp=1.0, top_k=10).item() for _ in range(n_samples)]
class_samples = [TokenSampler.top_k_sampling(logits, k=10, temperature=1.0).item()
                 for _ in range(n_samples)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(compact_samples, bins=30, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].set_title('sampling.sample(temp=1.0, top_k=10)')
axes[0].set_xlabel('Token ID')
axes[0].set_ylabel('Frequency')

axes[1].hist(class_samples, bins=30, alpha=0.7, color='coral', edgecolor='black')
axes[1].set_title('TokenSampler.top_k_sampling(k=10, T=1.0)')
axes[1].set_xlabel('Token ID')
axes[1].set_ylabel('Frequency')

plt.suptitle('Compact vs Class-Based Sampler (same parameters)', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Compact sampler unique tokens: {len(set(compact_samples))}")
print(f"Class sampler unique tokens:   {len(set(class_samples))}")

Both implementations produce the same set of 10 candidate tokens.
The compact version is useful for inference loops where readability and
minimal overhead matter.

---
## 10. Combining Strategies

In practice, strategies are often combined. For example, OpenAI's API applies
top-p filtering *after* temperature scaling. Let us see how combinations interact.

In [ ]:
combos = [
    {'label': 'T=1.0 only',         'temp': 1.0, 'k': None, 'p': None},
    {'label': 'T=0.7 + top-k=40',   'temp': 0.7, 'k': 40,  'p': None},
    {'label': 'T=0.9 + top-p=0.9',  'temp': 0.9, 'k': None, 'p': 0.9},
    {'label': 'T=0.7 + top-k=50 + top-p=0.9', 'temp': 0.7, 'k': 50, 'p': 0.9},
]

print("Combined strategy sampling (5 samples each):")
print("=" * 65)
for combo in combos:
    samples = []
    for _ in range(5):
        if combo['k'] is not None and combo['p'] is not None:
            # Apply top-k first, then top-p
            tok = TokenSampler.top_k_sampling(logits, k=combo['k'], temperature=combo['temp'])
        elif combo['k'] is not None:
            tok = TokenSampler.top_k_sampling(logits, k=combo['k'], temperature=combo['temp'])
        elif combo['p'] is not None:
            tok = TokenSampler.top_p_sampling(logits, p=combo['p'], temperature=combo['temp'])
        else:
            tok = TokenSampler.temperature_sampling(logits, temperature=combo['temp'])
        samples.append(tok.item())
    print(f"  {combo['label']:<35} -> tokens: {samples}")

---
## 11. Practical Recommendations

| Use Case | Recommended Settings | Why |
|---|---|---|
| **Code generation** | T=0.2, top-p=0.95 | Correctness matters; low diversity needed |
| **Conversational AI** | T=0.7, top-p=0.9 | Balanced fluency and variety |
| **Creative writing** | T=1.0-1.2, top-p=0.95 | Encourage novel phrasing |
| **Brainstorming** | T=1.3-1.5, top-k=50 | Maximize diversity, accept some noise |
| **Factual Q&A** | T=0.0-0.3 (greedy) | Minimize hallucination |
| **Translation** | T=0.5, top-k=10 | Constrained but not fully greedy |

**General guidelines:**
- Start with **T=0.7, top-p=0.9** as a baseline and adjust from there
- **Top-p is generally preferred over top-k** because it adapts to distribution shape
- **Never use high temperature without truncation** (top-k or top-p) -- the tail tokens are usually garbage
- Combining temperature with top-p is the most common production configuration

---
## 12. Summary of Ablation Results

In [ ]:
# Print numerical ablation results for reference
temp_results = analyzer.ablate_temperature()
topk_results = analyzer.ablate_top_k()
topp_results = analyzer.ablate_top_p()

print("Temperature Ablation:")
print(f"  {'Temp':<8} {'Entropy':<10} {'Max Prob':<10} {'Sparsity':<10}")
for i in range(len(temp_results['temp'])):
    print(f"  {temp_results['temp'][i]:<8} "
          f"{temp_results['entropy'][i]:<10.3f} "
          f"{temp_results['max_prob'][i]:<10.4f} "
          f"{temp_results['sparsity'][i]:<10.3f}")

print("\nTop-K Ablation:")
print(f"  {'K':<8} {'Entropy':<10} {'Coverage':<10}")
for i in range(len(topk_results['k'])):
    print(f"  {topk_results['k'][i]:<8} "
          f"{topk_results['entropy'][i]:<10.3f} "
          f"{topk_results['coverage'][i]:<10.4f}")

print("\nTop-P Ablation:")
print(f"  {'P':<8} {'Entropy':<10} {'Tokens':<10}")
for i in range(len(topp_results['p'])):
    print(f"  {topp_results['p'][i]:<8} "
          f"{topp_results['entropy'][i]:<10.3f} "
          f"{topp_results['num_tokens'][i]:<10}")

---
## Key Insights

1. **Temperature** is a global knob that reshapes the entire distribution. Low temperature sharpens (more greedy), high temperature flattens (more random). It is the simplest and most widely used parameter.

2. **Top-k** provides a hard cutoff: exactly k tokens survive, regardless of how the probability mass is distributed. This makes it predictable but inflexible -- it cannot adapt to peaked vs flat distributions.

3. **Top-p (nucleus) sampling** is adaptive: it selects the smallest set of tokens that cover p% of the probability mass. For peaked distributions this might be 3 tokens; for flat distributions, 200. This adaptivity makes it the preferred truncation method in modern systems.

4. **Entropy** is the unifying metric: all three strategies reduce entropy compared to the raw distribution. Lower entropy means more deterministic output. The ablation plots show exactly how each parameter maps to entropy.

5. **Combining strategies** is standard practice. Temperature + top-p is the most common production configuration (e.g., T=0.7, p=0.9). Temperature controls the overall randomness, while top-p removes the dangerous tail.

6. **Sparsity reveals the tail problem**: even at T=1.0, over 99% of tokens have probability below 1%. Sampling without truncation means occasionally generating these near-zero-probability tokens, which are almost always incoherent.